# 01b — Scrape ICWSM & JCDL Award Data

Scrapes Best Paper and Test of Time award winners from:
- **ICWSM**: `icwsm.org/awards/`
- **JCDL**: ACM Digital Library proceedings pages

Output columns match `huang_awards_cleaned.csv` schema, plus `award_type`:
```
year | conference | paper_title | paper_url | authors | award_type
```

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (research scraper; thesis project)"
}

## 1. Scrape ICWSM Awards

In [ ]:
def scrape_icwsm_awards():
    url = "https://icwsm.org/awards/"
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    records = []
    current_award_type = None
    current_year = None

    for tag in soup.find_all(["h2", "h3", "h4", "p", "li", "strong"]):
        text = tag.get_text(separator=" ", strip=True)

        # Detect award type headings
        if re.search(r"test of time", text, re.IGNORECASE):
            current_award_type = "Test of Time"
        elif re.search(r"best paper", text, re.IGNORECASE):
            current_award_type = "Best Paper"
        elif re.search(r"honorable mention", text, re.IGNORECASE):
            current_award_type = "Honorable Mention"

        # Detect year sub-headings
        year_match = re.search(r"\b(20\d{2})\b", text)
        if year_match and tag.name in ["h2", "h3", "h4", "strong"]:
            current_year = int(year_match.group(1))

        # Extract paper entries
        if tag.name in ["p", "li"] and current_year and current_award_type:
            title_tag = tag.find(["em", "i"])
            title = title_tag.get_text(strip=True) if title_tag else None

            link_tag = tag.find("a", href=True)
            paper_url = link_tag["href"] if link_tag else ""

            if title:
                full_text = text.replace(title, "").strip().strip(",.-").strip()
                records.append({
                    "year": current_year,
                    "conference": "ICWSM",
                    "paper_title": title,
                    "paper_url": paper_url,
                    "authors": full_text,
                    "award_type": current_award_type
                })

    return pd.DataFrame(records)


df_icwsm = scrape_icwsm_awards()
print(f"ICWSM records scraped: {len(df_icwsm)}")
df_icwsm.head(10)

## 2. Inspect Raw HTML (debug helper)

Run this cell to inspect the actual page structure if the scraper above misses entries.

In [ ]:
resp = requests.get("https://icwsm.org/awards/", headers=HEADERS, timeout=15)
soup = BeautifulSoup(resp.text, "html.parser")

for tag in list(soup.find_all(True))[:200]:
    if tag.name in ["h1", "h2", "h3", "h4", "p", "li", "strong", "em"]:
        print(f"<{tag.name}> {tag.get_text(strip=True)[:120]}")

## 3. Scrape JCDL Awards (ACM DL)

JCDL doesn't maintain a clean awards page, so we scrape the ACM DL proceedings for each year
and filter for award-tagged papers.

In [ ]:
def get_jcdl_proceedings_links():
    proceedings_url = "https://dl.acm.org/conference/jcdl/proceedings"
    resp = requests.get(proceedings_url, headers=HEADERS, timeout=20)
    soup = BeautifulSoup(resp.text, "html.parser")

    print(f"Page title: {soup.title.get_text() if soup.title else 'N/A'}")

    proc_links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/doi/proceedings/" in href or ("/doi/10." in href and "jcdl" in href.lower()):
            year_m = re.search(r"(20\d{2}|199\d)", a.get_text())
            if year_m:
                full_url = "https://dl.acm.org" + href if href.startswith("/") else href
                proc_links.append((int(year_m.group()), full_url))

    proc_links = list(dict.fromkeys(proc_links))  # deduplicate
    print(f"Found {len(proc_links)} proceedings links")
    return proc_links


proc_links = get_jcdl_proceedings_links()
proc_links[:10]

In [ ]:
def get_jcdl_awards_from_proceedings(proc_links):
    records = []

    for year, url in sorted(proc_links):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=20)
            soup = BeautifulSoup(resp.text, "html.parser")
            time.sleep(1.5)

            award_items = soup.find_all(class_=re.compile(r"award", re.IGNORECASE))

            for item in award_items:
                paper_block = item.find_parent(class_=re.compile(r"issue-item|article|result"))
                if not paper_block:
                    paper_block = item.find_parent("li")
                if not paper_block:
                    continue

                title_tag = paper_block.find(["h5", "h4", "h3"], class_=re.compile(r"title", re.IGNORECASE))
                if not title_tag:
                    title_tag = paper_block.find("a", href=re.compile(r"/doi/"))

                title = title_tag.get_text(strip=True) if title_tag else ""
                paper_url = ""
                if title_tag and title_tag.name == "a":
                    href = title_tag.get("href", "")
                    paper_url = "https://dl.acm.org" + href if href.startswith("/") else href

                authors_block = paper_block.find(class_=re.compile(r"author|contrib", re.IGNORECASE))
                authors = authors_block.get_text(separator=", ", strip=True) if authors_block else ""

                award_text = item.get_text(strip=True)
                if re.search(r"test of time", award_text, re.IGNORECASE):
                    award_type = "Test of Time"
                elif re.search(r"honorable", award_text, re.IGNORECASE):
                    award_type = "Honorable Mention"
                else:
                    award_type = "Best Paper"

                if title:
                    records.append({
                        "year": year,
                        "conference": "JCDL",
                        "paper_title": title,
                        "paper_url": paper_url,
                        "authors": authors,
                        "award_type": award_type
                    })

            print(f"{year}: {len([r for r in records if r['year'] == year])} award papers found")

        except Exception as e:
            print(f"{year}: ERROR — {e}")
            continue

    return pd.DataFrame(records)


df_jcdl = get_jcdl_awards_from_proceedings(proc_links)
print(f"\nTotal JCDL records: {len(df_jcdl)}")
df_jcdl.head(10)

## 4. Debug — Inspect One Proceedings Page

If ACM DL returns 0 award results (JS-rendered), inspect the raw HTML here to adjust selectors.

In [ ]:
if proc_links:
    test_year, test_url = sorted(proc_links)[-1]
    resp = requests.get(test_url, headers=HEADERS, timeout=20)
    soup_test = BeautifulSoup(resp.text, "html.parser")
    print(f"Year: {test_year}")
    print(f"Page title: {soup_test.title.get_text() if soup_test.title else 'N/A'}")
    print(f"Total tags: {len(soup_test.find_all(True))}")
    award_els = soup_test.find_all(class_=re.compile(r"award", re.IGNORECASE))
    print(f"Award-tagged elements: {len(award_els)}")
    for el in award_els[:5]:
        print(el)

## 5. Combine & Save

In [ ]:
df_all = pd.concat([df_icwsm, df_jcdl], ignore_index=True)
df_all = df_all[["year", "conference", "paper_title", "paper_url", "authors", "award_type"]]

df_all["paper_title"] = df_all["paper_title"].str.strip()
df_all["authors"] = df_all["authors"].str.strip()
df_all = df_all[df_all["paper_title"].notna() & (df_all["paper_title"] != "")]
df_all = df_all.drop_duplicates(subset=["paper_title", "year", "conference"])

print(f"Total records: {len(df_all)}")
print(df_all["conference"].value_counts())
print(df_all["award_type"].value_counts())
df_all.head()

In [ ]:
df_icwsm.to_csv("../data/raw/icwsm_awards_raw.csv", index=False)
df_jcdl.to_csv("../data/raw/jcdl_awards_raw.csv", index=False)
df_all.to_csv("../data/raw/icwsm_jcdl_awards_combined.csv", index=False)

print("Saved:")
print("  data/raw/icwsm_awards_raw.csv")
print("  data/raw/jcdl_awards_raw.csv")
print("  data/raw/icwsm_jcdl_awards_combined.csv")

## 6. OpenAlex Fields Preview

Quick test query to confirm which fields are available before running the enrichment notebook.

In [ ]:
def query_openalex(title, mailto="thesis@example.com"):
    url = "https://api.openalex.org/works"
    params = {"search": title, "per_page": 1, "mailto": mailto}
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    return results[0] if results else None


if len(df_all) > 0:
    test_title = df_all.iloc[0]["paper_title"]
    print(f"Querying OpenAlex for: {test_title}")
    result = query_openalex(test_title)

    if result:
        print("\nAvailable top-level fields:")
        for k, v in result.items():
            print(f"  {k}: {str(v)[:80]}")
    else:
        print("No result found — check title or network")

In [ ]:
# Fields we will extract in the enrichment step (04_matching_to_openalex.ipynb)
OPENALEX_FIELDS = [
    "id",                # OpenAlex ID  e.g. W2741809807
    "doi",               # DOI
    "title",             # canonical title
    "publication_year",  # year
    "cited_by_count",    # total citations
    "authorships",       # [{author: {id, display_name}, institutions: [...]}]
    "primary_location",  # venue info
    "concepts",          # topic tags with scores
    "open_access",       # OA status
    "counts_by_year",    # citation trajectory (year-by-year) <-- key for trajectories
]

print("Fields to extract in enrichment notebook:")
for f in OPENALEX_FIELDS:
    print(f"  - {f}")